In [2]:
# ============================================
# Startup Cell: Mount Drive + Import Config
# ============================================

from google.colab import drive
drive.mount("/content/drive")

import sys
import os

# -------------------------------------------------
# Project base directory (Drive)
# -------------------------------------------------
BASE_DRIVE_DIR = "/content/drive/MyDrive/DIP_Project"

# -------------------------------------------------
# Add src/ to Python path and import config
# -------------------------------------------------
sys.path.append(f"{BASE_DRIVE_DIR}/src")

from project_config import *

# -------------------------------------------------
# Basic environment confirmation
# -------------------------------------------------
print("Drive mounted successfully.")
print(f"BASE_DIR: {BASE_DIR}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"METADATA_DIR: {METADATA_DIR}")
print(f"LOCAL_WORK_DIR: {LOCAL_WORK_DIR}")



Mounted at /content/drive
Drive mounted successfully.
BASE_DIR: /content/drive/MyDrive/DIP_Project
DATA_DIR: /content/drive/MyDrive/DIP_Project/data
METADATA_DIR: /content/drive/MyDrive/DIP_Project/data/metadata
LOCAL_WORK_DIR: /content


In [3]:
# ============================================
# Cell 0: Notebook Overview
# ============================================
# Purpose:
#   This notebook builds final fixed-length feature vectors for both the
#   training and test subsets by combining gradient-based, spatial, and
#   frequency-domain feature CSV files into classifier-ready tables.
#
# Inputs:
#   The notebook expects:
#     - subset-specific feature CSV files stored in:
#         data/metadata/
#       including:
#         train_gradient_features.csv
#         train_spatial_features.csv
#         train_frequency_features.csv
#         test_gradient_features.csv
#         test_spatial_features.csv
#         test_frequency_features.csv
#
# Assumptions:
#   - Feature extraction has already been completed for both subsets.
#   - Each input CSV contains one row per image for its corresponding
#     subset.
#   - Metadata columns are repeated in each feature CSV.
#   - Rows align across all three CSV files for the same images within
#     each subset.
#   - The notebook uses project_config.py as the central source for
#     directory paths, file names, and shared project constants.
#   - This notebook performs feature-vector assembly only.
#
# What the notebook does:
#   Startup Cell:
#     Mount Google Drive, import project_config.py, and initialize
#     the notebook environment.
#
#   Cell 1:
#     Import required libraries for CSV loading, validation, and
#     feature-vector construction.
#
#   Cell 2:
#     Define input feature CSV paths, output feature-vector CSV paths,
#     expected row counts, and feature column groupings for both the
#     training and test subsets.
#
#   Cell 3:
#     Verify required input CSV files exist and load the feature CSVs
#     for both subsets.
#
#   Cell 4:
#     Validate that the gradient, spatial, and frequency feature CSVs
#     align correctly within each subset:
#       - confirm expected row counts
#       - confirm metadata columns match
#       - confirm filenames align row-by-row
#       - confirm subset labels are correct
#
#   Cell 5:
#     Define metadata and feature column lists used to assemble the
#     final feature-vector tables.
#
#   Cell 6:
#     Build the final training and test feature-vector tables by:
#       - retaining one copy of the metadata columns
#       - appending gradient-based feature columns
#       - appending spatial feature columns
#       - appending frequency-domain feature columns
#
#   Cell 7:
#     Validate the final training and test feature-vector tables:
#       - confirm feature counts
#       - confirm row counts
#       - confirm no unexpected missing values
#
#   Cell 8:
#     Save:
#       - train_feature_vectors.csv
#       - test_feature_vectors.csv
#
#   Cell 9:
#     Reload and verify the saved training and test feature-vector CSVs.
#
# Outputs:
#   Primary outputs:
#     - train_feature_vectors.csv
#     - test_feature_vectors.csv
#
#   Final feature-vector contents:
#     - metadata columns
#     - 8 gradient-based features
#     - 9 spatial features
#     - 8 frequency-domain features
#     - total DIP features: 25
#
# Notes:
#   - This notebook builds both training and test feature-vector files
#     in a single run.
#   - No normalization is performed here.
#   - No classifier training is performed here.
#   - The training and test feature-vector files must remain strictly
#     separate. The training set is used for model development, while
#     the test set is reserved for final evaluation only.
# ============================================

print("Notebook overview loaded.")



Notebook overview loaded.


In [4]:
# ============================================
# Cell 1: Imports
# ============================================

import os
import pandas as pd
import numpy as np



In [5]:
# ============================================
# Cell 2: Define Input and Output Paths
# ============================================

# -------------------------------------------------
# Define input paths (from metadata directory)
# -------------------------------------------------
TRAIN_GRADIENT_CSV  = os.path.join(METADATA_DIR, "train_gradient_features.csv")
TRAIN_SPATIAL_CSV   = os.path.join(METADATA_DIR, "train_spatial_features.csv")
TRAIN_FREQUENCY_CSV = os.path.join(METADATA_DIR, "train_frequency_features.csv")

TEST_GRADIENT_CSV   = os.path.join(METADATA_DIR, "test_gradient_features.csv")
TEST_SPATIAL_CSV    = os.path.join(METADATA_DIR, "test_spatial_features.csv")
TEST_FREQUENCY_CSV  = os.path.join(METADATA_DIR, "test_frequency_features.csv")

# -------------------------------------------------
# Define output paths
# -------------------------------------------------
TRAIN_OUTPUT_CSV = os.path.join(METADATA_DIR, TRAIN_FEATURE_VECTOR_FILENAME)
TEST_OUTPUT_CSV  = os.path.join(METADATA_DIR, TEST_FEATURE_VECTOR_FILENAME)

# -------------------------------------------------
# Expected row counts (from config)
# -------------------------------------------------
EXPECTED_ROWS = {
    TRAIN_SUBSET: TRAIN_IMAGES,
    TEST_SUBSET: TEST_IMAGES
}

# -------------------------------------------------
# Display configuration
# -------------------------------------------------
print("Training input files:")
print("  GRADIENT  =", TRAIN_GRADIENT_CSV)
print("  SPATIAL   =", TRAIN_SPATIAL_CSV)
print("  FREQUENCY =", TRAIN_FREQUENCY_CSV)
print("  OUTPUT    =", TRAIN_OUTPUT_CSV)
print("  EXPECTED ROWS =", EXPECTED_ROWS[TRAIN_SUBSET])

print("\nTest input files:")
print("  GRADIENT  =", TEST_GRADIENT_CSV)
print("  SPATIAL   =", TEST_SPATIAL_CSV)
print("  FREQUENCY =", TEST_FREQUENCY_CSV)
print("  OUTPUT    =", TEST_OUTPUT_CSV)
print("  EXPECTED ROWS =", EXPECTED_ROWS[TEST_SUBSET])



Training input files:
  GRADIENT  = /content/drive/MyDrive/DIP_Project/data/metadata/train_gradient_features.csv
  SPATIAL   = /content/drive/MyDrive/DIP_Project/data/metadata/train_spatial_features.csv
  FREQUENCY = /content/drive/MyDrive/DIP_Project/data/metadata/train_frequency_features.csv
  OUTPUT    = /content/drive/MyDrive/DIP_Project/data/metadata/train_feature_vectors.csv
  EXPECTED ROWS = 14400

Test input files:
  GRADIENT  = /content/drive/MyDrive/DIP_Project/data/metadata/test_gradient_features.csv
  SPATIAL   = /content/drive/MyDrive/DIP_Project/data/metadata/test_spatial_features.csv
  FREQUENCY = /content/drive/MyDrive/DIP_Project/data/metadata/test_frequency_features.csv
  OUTPUT    = /content/drive/MyDrive/DIP_Project/data/metadata/test_feature_vectors.csv
  EXPECTED ROWS = 3600


In [6]:
# ============================================
# Cell 3: Load Feature CSVs
# ============================================

train_paths = [
    TRAIN_GRADIENT_CSV,
    TRAIN_SPATIAL_CSV,
    TRAIN_FREQUENCY_CSV,
]

test_paths = [
    TEST_GRADIENT_CSV,
    TEST_SPATIAL_CSV,
    TEST_FREQUENCY_CSV,
]

for path in train_paths + test_paths:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required file: {path}")

# -------------------------------------------------
# Load training feature CSVs
# -------------------------------------------------
df_grad_train = pd.read_csv(TRAIN_GRADIENT_CSV)
df_spat_train = pd.read_csv(TRAIN_SPATIAL_CSV)
df_freq_train = pd.read_csv(TRAIN_FREQUENCY_CSV)

# -------------------------------------------------
# Load test feature CSVs
# -------------------------------------------------
df_grad_test = pd.read_csv(TEST_GRADIENT_CSV)
df_spat_test = pd.read_csv(TEST_SPATIAL_CSV)
df_freq_test = pd.read_csv(TEST_FREQUENCY_CSV)

print("Training CSV shapes:")
print("  Gradient : ", df_grad_train.shape)
print("  Spatial  : ", df_spat_train.shape)
print("  Frequency: ", df_freq_train.shape)

print("\nTest CSV shapes:")
print("  Gradient : ", df_grad_test.shape)
print("  Spatial  : ", df_spat_test.shape)
print("  Frequency: ", df_freq_test.shape)

print("\nExpected rows:")
print("  Train:", EXPECTED_ROWS[TRAIN_SUBSET])
print("  Test :", EXPECTED_ROWS[TEST_SUBSET])


Training CSV shapes:
  Gradient :  (14400, 12)
  Spatial  :  (14400, 13)
  Frequency:  (14400, 12)

Test CSV shapes:
  Gradient :  (3600, 12)
  Spatial  :  (3600, 13)
  Frequency:  (3600, 12)

Expected rows:
  Train: 14400
  Test : 3600


In [7]:
# ============================================
# Cell 4: Validate Row Counts and Alignment
# ============================================

metadata_columns = ["filename", "source_dataset", "class_label", "subset"]

def validate_subset_alignment(df_grad, df_spat, df_freq, expected_rows, subset_name):
    # -------------------------------------------------
    # Validate row counts
    # -------------------------------------------------
    if len(df_grad) != expected_rows:
        raise ValueError(
            f"{subset_name} gradient CSV row count mismatch: expected {expected_rows}, got {len(df_grad)}"
        )

    if len(df_spat) != expected_rows:
        raise ValueError(
            f"{subset_name} spatial CSV row count mismatch: expected {expected_rows}, got {len(df_spat)}"
        )

    if len(df_freq) != expected_rows:
        raise ValueError(
            f"{subset_name} frequency CSV row count mismatch: expected {expected_rows}, got {len(df_freq)}"
        )

    # -------------------------------------------------
    # Validate metadata alignment
    # -------------------------------------------------
    for col in metadata_columns:
        if not df_grad[col].equals(df_spat[col]):
            raise ValueError(
                f"{subset_name} metadata mismatch between gradient and spatial CSVs in column: {col}"
            )
        if not df_grad[col].equals(df_freq[col]):
            raise ValueError(
                f"{subset_name} metadata mismatch between gradient and frequency CSVs in column: {col}"
            )

    print(f"PASS: {subset_name} input CSVs have the expected row count")
    print(f"PASS: {subset_name} metadata columns align across all three CSVs")


# -------------------------------------------------
# Validate training subset
# -------------------------------------------------
validate_subset_alignment(
    df_grad_train,
    df_spat_train,
    df_freq_train,
    EXPECTED_ROWS[TRAIN_SUBSET],
    TRAIN_SUBSET
)

# -------------------------------------------------
# Validate test subset
# -------------------------------------------------
validate_subset_alignment(
    df_grad_test,
    df_spat_test,
    df_freq_test,
    EXPECTED_ROWS[TEST_SUBSET],
    TEST_SUBSET
)



PASS: train input CSVs have the expected row count
PASS: train metadata columns align across all three CSVs
PASS: test input CSVs have the expected row count
PASS: test metadata columns align across all three CSVs


In [8]:
# ============================================
# Cell 5: Define Metadata and Feature Columns
# ============================================

metadata_cols = [
    "filename",
    "class_label",
    "source_dataset",
    "subset"
]

gradient_feature_cols = [
    "Mean Gradient",
    "Std Gradient",
    "Max Gradient",
    "Gradient Entropy",
    "Edge Density",
    "Orientation Mean",
    "Orientation Std",
    "Orientation Entropy"
]

spatial_feature_cols = [
    "Global Entropy",
    "Local Entropy Mean",
    "Local Entropy Std",
    "Intensity Mean",
    "Intensity Std",
    "Laplacian Variance",
    "Patch Variance Mean",
    "Patch Variance Std",
    "Noise Residual Energy"
]

frequency_feature_cols = [
    "Low Frequency Energy Ratio",
    "High Frequency Energy Ratio",
    "Radial Mean",
    "Radial Std",
    "Radial Entropy",
    "Spectral Centroid",
    "Spectral Bandwidth",
    "Log Spectrum Std"
]

print("Metadata columns: ", len(metadata_cols))
print("Gradient features:", len(gradient_feature_cols))
print("Spatial features: ", len(spatial_feature_cols))
print("Frequency features:", len(frequency_feature_cols))
print("Total DIP features:",
      len(gradient_feature_cols)
    + len(spatial_feature_cols)
    + len(frequency_feature_cols))


Metadata columns:  4
Gradient features: 8
Spatial features:  9
Frequency features: 8
Total DIP features: 25


In [9]:
# ============================================
# Cell 6: Verify Required Columns Exist
# ============================================

def check_columns(df, required_cols, df_name):
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"{df_name} is missing columns: {missing}")
    print(f"{df_name}: all required columns present")


# -------------------------------------------------
# Training set checks
# -------------------------------------------------
check_columns(df_grad_train, metadata_cols + gradient_feature_cols, "Train Gradient CSV")
check_columns(df_spat_train, metadata_cols + spatial_feature_cols, "Train Spatial CSV")
check_columns(df_freq_train, metadata_cols + frequency_feature_cols, "Train Frequency CSV")


# -------------------------------------------------
# Test set checks
# -------------------------------------------------
check_columns(df_grad_test, metadata_cols + gradient_feature_cols, "Test Gradient CSV")
check_columns(df_spat_test, metadata_cols + spatial_feature_cols, "Test Spatial CSV")
check_columns(df_freq_test, metadata_cols + frequency_feature_cols, "Test Frequency CSV")



Train Gradient CSV: all required columns present
Train Spatial CSV: all required columns present
Train Frequency CSV: all required columns present
Test Gradient CSV: all required columns present
Test Spatial CSV: all required columns present
Test Frequency CSV: all required columns present


In [10]:
# ============================================
# Cell 7: Verify Metadata Alignment Across CSVs
# ============================================

def check_metadata_alignment(df_grad, df_spat, df_freq, subset_name):
    for col in metadata_cols:
        same_grad_spat = df_grad[col].equals(df_spat[col])
        same_grad_freq = df_grad[col].equals(df_freq[col])

        print(f"{subset_name} | {col}:")
        print(f"  Gradient vs Spatial:   {same_grad_spat}")
        print(f"  Gradient vs Frequency: {same_grad_freq}")

        if not same_grad_spat or not same_grad_freq:
            raise ValueError(f"{subset_name}: mismatch detected in metadata column: {col}")

    print(f"\nPASS: {subset_name} metadata columns align across CSVs\n")


# -------------------------------------------------
# Training set
# -------------------------------------------------
check_metadata_alignment(
    df_grad_train,
    df_spat_train,
    df_freq_train,
    TRAIN_SUBSET
)

# -------------------------------------------------
# Test set
# -------------------------------------------------
check_metadata_alignment(
    df_grad_test,
    df_spat_test,
    df_freq_test,
    TEST_SUBSET
)



train | filename:
  Gradient vs Spatial:   True
  Gradient vs Frequency: True
train | class_label:
  Gradient vs Spatial:   True
  Gradient vs Frequency: True
train | source_dataset:
  Gradient vs Spatial:   True
  Gradient vs Frequency: True
train | subset:
  Gradient vs Spatial:   True
  Gradient vs Frequency: True

PASS: train metadata columns align across CSVs

test | filename:
  Gradient vs Spatial:   True
  Gradient vs Frequency: True
test | class_label:
  Gradient vs Spatial:   True
  Gradient vs Frequency: True
test | source_dataset:
  Gradient vs Spatial:   True
  Gradient vs Frequency: True
test | subset:
  Gradient vs Spatial:   True
  Gradient vs Frequency: True

PASS: test metadata columns align across CSVs



In [11]:
# ============================================
# Cell 8: Verify Subset Labels Match Expected Subsets
# ============================================

def check_subset_labels(df_grad, df_spat, df_freq, expected_subset_value):
    unique_subsets_grad = sorted(df_grad["subset"].dropna().unique())
    unique_subsets_spat = sorted(df_spat["subset"].dropna().unique())
    unique_subsets_freq = sorted(df_freq["subset"].dropna().unique())

    print(f"{expected_subset_value} | Gradient subset values: ", unique_subsets_grad)
    print(f"{expected_subset_value} | Spatial subset values:  ", unique_subsets_spat)
    print(f"{expected_subset_value} | Frequency subset values:", unique_subsets_freq)

    if unique_subsets_grad != [expected_subset_value]:
        raise ValueError(
            f"{expected_subset_value} gradient CSV subset mismatch: "
            f"expected [{expected_subset_value}], got {unique_subsets_grad}"
        )

    if unique_subsets_spat != [expected_subset_value]:
        raise ValueError(
            f"{expected_subset_value} spatial CSV subset mismatch: "
            f"expected [{expected_subset_value}], got {unique_subsets_spat}"
        )

    if unique_subsets_freq != [expected_subset_value]:
        raise ValueError(
            f"{expected_subset_value} frequency CSV subset mismatch: "
            f"expected [{expected_subset_value}], got {unique_subsets_freq}"
        )

    print(f"PASS: subset labels match {expected_subset_value}\n")


# -------------------------------------------------
# Training set
# -------------------------------------------------
check_subset_labels(
    df_grad_train,
    df_spat_train,
    df_freq_train,
    TRAIN_SUBSET
)

# -------------------------------------------------
# Test set
# -------------------------------------------------
check_subset_labels(
    df_grad_test,
    df_spat_test,
    df_freq_test,
    TEST_SUBSET
)



train | Gradient subset values:  ['train']
train | Spatial subset values:   ['train']
train | Frequency subset values: ['train']
PASS: subset labels match train

test | Gradient subset values:  ['test']
test | Spatial subset values:   ['test']
test | Frequency subset values: ['test']
PASS: subset labels match test



In [12]:
# ============================================
# Cell 9: Build Final Feature Vector DataFrames
# ============================================

# -------------------------------------------------
# Build training feature-vector table
# -------------------------------------------------
df_feature_vectors_train = pd.concat(
    [
        df_grad_train[metadata_cols],
        df_grad_train[gradient_feature_cols],
        df_spat_train[spatial_feature_cols],
        df_freq_train[frequency_feature_cols]
    ],
    axis=1
)

# -------------------------------------------------
# Build test feature-vector table
# -------------------------------------------------
df_feature_vectors_test = pd.concat(
    [
        df_grad_test[metadata_cols],
        df_grad_test[gradient_feature_cols],
        df_spat_test[spatial_feature_cols],
        df_freq_test[frequency_feature_cols]
    ],
    axis=1
)

print("Training feature vector shape:", df_feature_vectors_train.shape)
display(df_feature_vectors_train.head())

print("\nTest feature vector shape:", df_feature_vectors_test.shape)
display(df_feature_vectors_test.head())



Training feature vector shape: (14400, 29)


,filename,class_label,source_dataset,subset,Mean Gradient,Std Gradient,Max Gradient,Gradient Entropy,Edge Density,Orientation Mean,...,Patch Variance Std,Noise Residual Energy,Low Frequency Energy Ratio,High Frequency Energy Ratio,Radial Mean,Radial Std,Radial Entropy,Spectral Centroid,Spectral Bandwidth,Log Spectrum Std
0,rl_imgn_002320.png,rl,ImageNet_1K_256,train,113.284256,148.626465,988.380493,4.200053,0.133926,0.009213,...,1845.180777,51.161957,0.977534,0.005830,4.172419e+11,5.524948e+12,0.049157,0.042217,0.785720,0.944199
1,rl_coco_001397.png,rl,MS_COCO_2017,train,58.244781,91.992287,943.153198,3.224193,0.104218,0.003133,...,1525.370321,26.302368,0.975984,0.004450,8.717484e+10,1.120272e+12,0.098270,0.103196,0.981119,1.096229
2,rl_imgn_001958.png,rl,ImageNet_1K_256,train,99.089622,129.458405,955.745789,4.071966,0.119492,-0.031641,...,1353.469364,47.263107,0.970208,0.007792,2.700694e+11,3.598769e+12,0.049157,0.033637,0.839125,0.934185
3,rl_coco_000800.png,rl,MS_COCO_2017,train,76.498199,114.998993,924.299744,3.541116,0.123779,0.210707,...,1453.744813,35.510101,0.992152,0.001953,6.987824e+11,9.309489e+12,0.049157,0.018659,0.438470,0.960530
4,ai_mj_002892.png,ai,Midjourney,train,67.460518,74.748627,801.519836,3.823527,0.113922,-0.004285,...,971.494479,27.173721,0.993974,0.000909,3.318048e+11,4.394753e+12,0.049157,0.035712,0.498081,1.125460



Test feature vector shape: (3600, 29)


,filename,class_label,source_dataset,subset,Mean Gradient,Std Gradient,Max Gradient,Gradient Entropy,Edge Density,Orientation Mean,...,Patch Variance Std,Noise Residual Energy,Low Frequency Energy Ratio,High Frequency Energy Ratio,Radial Mean,Radial Std,Radial Entropy,Spectral Centroid,Spectral Bandwidth,Log Spectrum Std
0,rl_coco_001786.png,rl,MS_COCO_2017,test,94.913315,84.709961,949.826294,4.032700,0.116837,-0.006126,...,739.228677,54.917587,0.982947,0.003721,3.012395e+11,4.009858e+12,0.049157,0.022613,0.598543,0.984959
1,rl_coco_001292.png,rl,MS_COCO_2017,test,111.519829,115.061188,993.909424,4.280079,0.125336,-0.024534,...,1405.213121,59.734268,0.974834,0.006682,2.891114e+11,3.824329e+12,0.049157,0.043467,0.820382,0.918321
2,ai_sdxl_002062.png,ai,SDXL_Generated_10K,test,51.687538,102.624313,1013.039978,2.621670,0.102768,-0.072338,...,1363.939551,20.048294,0.994355,0.000774,5.191253e+11,6.932645e+12,0.049157,0.017100,0.406515,1.143770
3,ai_sdxl_000022.png,ai,SDXL_Generated_10K,test,91.249680,78.876884,525.077148,4.872271,0.143631,0.227782,...,495.516592,46.851273,0.986175,0.002371,2.411244e+11,3.228151e+12,0.049157,0.024168,0.605342,1.276786
4,ai_diff_002198.png,ai,DiffusionDB,test,70.792397,121.394005,957.306641,3.184196,0.114090,0.227915,...,1573.042934,33.158493,0.984120,0.003585,3.451115e+11,4.557373e+12,0.049157,0.037299,0.673160,0.938902


In [13]:
# ============================================
# Cell 10: Validate Final Feature Vector Tables
# ============================================

expected_feature_count = (
    len(gradient_feature_cols) +
    len(spatial_feature_cols) +
    len(frequency_feature_cols)
)

def validate_feature_vector_table(df_feature_vectors, expected_rows, subset_name):
    actual_feature_count = len(df_feature_vectors.columns) - len(metadata_cols)

    print(f"{subset_name} | Expected feature count: {expected_feature_count}")
    print(f"{subset_name} | Actual feature count:   {actual_feature_count}")

    if actual_feature_count != expected_feature_count:
        raise ValueError(f"{subset_name}: feature count mismatch")

    if len(df_feature_vectors) != expected_rows:
        raise ValueError(
            f"{subset_name}: final row count mismatch: "
            f"expected {expected_rows}, got {len(df_feature_vectors)}"
        )

    print(f"\n{subset_name} | Class counts:")
    print(df_feature_vectors["class_label"].value_counts())

    print(f"\n{subset_name} | Source counts:")
    print(df_feature_vectors["source_dataset"].value_counts())

    print(f"\n{subset_name} | Subset counts:")
    print(df_feature_vectors["subset"].value_counts())

    missing_counts = df_feature_vectors.isnull().sum()
    missing_counts = missing_counts[missing_counts > 0]

    print(f"\n{subset_name} | Missing values per column:")
    if len(missing_counts) == 0:
        print("None")
    else:
        print(missing_counts)

    print(f"\nPASS: {subset_name} feature vector table validated\n")


# -------------------------------------------------
# Validate training feature-vector table
# -------------------------------------------------
validate_feature_vector_table(
    df_feature_vectors_train,
    EXPECTED_ROWS[TRAIN_SUBSET],
    TRAIN_SUBSET
)

# -------------------------------------------------
# Validate test feature-vector table
# -------------------------------------------------
validate_feature_vector_table(
    df_feature_vectors_test,
    EXPECTED_ROWS[TEST_SUBSET],
    TEST_SUBSET
)



train | Expected feature count: 25
train | Actual feature count:   25

train | Class counts:
class_label
rl    7200
ai    7200
Name: count, dtype: int64

train | Source counts:
source_dataset
ImageNet_1K_256       2400
MS_COCO_2017          2400
Midjourney            2400
OpenImages            2400
SDXL_Generated_10K    2400
DiffusionDB           2400
Name: count, dtype: int64

train | Subset counts:
subset
train    14400
Name: count, dtype: int64

train | Missing values per column:
None

PASS: train feature vector table validated

test | Expected feature count: 25
test | Actual feature count:   25

test | Class counts:
class_label
rl    1800
ai    1800
Name: count, dtype: int64

test | Source counts:
source_dataset
MS_COCO_2017          600
SDXL_Generated_10K    600
DiffusionDB           600
OpenImages            600
Midjourney            600
ImageNet_1K_256       600
Name: count, dtype: int64

test | Subset counts:
subset
test    3600
Name: count, dtype: int64

test | Missing values 

In [14]:
# ============================================
# Cell 11: Save Final Feature Vectors
# ============================================

# -------------------------------------------------
# Save training feature vectors
# -------------------------------------------------
df_feature_vectors_train.to_csv(TRAIN_OUTPUT_CSV, index=False)
print(f"Saved: {TRAIN_OUTPUT_CSV}")
print("Train shape:", df_feature_vectors_train.shape)

# -------------------------------------------------
# Save test feature vectors
# -------------------------------------------------
df_feature_vectors_test.to_csv(TEST_OUTPUT_CSV, index=False)
print(f"Saved: {TEST_OUTPUT_CSV}")
print("Test shape:", df_feature_vectors_test.shape)



Saved: /content/drive/MyDrive/DIP_Project/data/metadata/train_feature_vectors.csv
Train shape: (14400, 29)
Saved: /content/drive/MyDrive/DIP_Project/data/metadata/test_feature_vectors.csv
Test shape: (3600, 29)


In [15]:
# ============================================
# Cell 12: Quick Sanity Check
# ============================================

print("Train - First 5 filenames:")
print(df_feature_vectors_train["filename"].head().to_list())

print("\nTrain - Column names:")
for col in df_feature_vectors_train.columns:
    print(col)

print("\n" + "="*50 + "\n")

print("Test - First 5 filenames:")
print(df_feature_vectors_test["filename"].head().to_list())

print("\nTest - Column names:")
for col in df_feature_vectors_test.columns:
    print(col)



Train - First 5 filenames:
['rl_imgn_002320.png', 'rl_coco_001397.png', 'rl_imgn_001958.png', 'rl_coco_000800.png', 'ai_mj_002892.png']

Train - Column names:
filename
class_label
source_dataset
subset
Mean Gradient
Std Gradient
Max Gradient
Gradient Entropy
Edge Density
Orientation Mean
Orientation Std
Orientation Entropy
Global Entropy
Local Entropy Mean
Local Entropy Std
Intensity Mean
Intensity Std
Laplacian Variance
Patch Variance Mean
Patch Variance Std
Noise Residual Energy
Low Frequency Energy Ratio
High Frequency Energy Ratio
Radial Mean
Radial Std
Radial Entropy
Spectral Centroid
Spectral Bandwidth
Log Spectrum Std


Test - First 5 filenames:
['rl_coco_001786.png', 'rl_coco_001292.png', 'ai_sdxl_002062.png', 'ai_sdxl_000022.png', 'ai_diff_002198.png']

Test - Column names:
filename
class_label
source_dataset
subset
Mean Gradient
Std Gradient
Max Gradient
Gradient Entropy
Edge Density
Orientation Mean
Orientation Std
Orientation Entropy
Global Entropy
Local Entropy Mean
Local 